# 02 · Maize baseline (transfer learning)

A deliberately small baseline: frozen MobileNetV3 backbone + a new classifier
head, then a short fine-tune. Copy this notebook per crop/model rather than
growing it into a monolith.

**Before you run:** point `DATA_ROOT` at a dataset with `train/` and `valid/`.
Training on CPU is viable for a baseline at this image size; expect a few
minutes per epoch and much faster on a GPU runtime.

In [ ]:
import sys
from pathlib import Path

def _notebook_dir() -> Path:
    """Locate notebooks/ whether Jupyter starts at the repo root, notebooks/, or deeper."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "notebooks" / "utils.py").exists():
            return candidate / "notebooks"
        if (candidate / "utils.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the repo (notebooks/utils.py not found)")


NOTEBOOK_DIR = _notebook_dir()
sys.path.insert(0, str(NOTEBOOK_DIR))

import utils

# ---- experiment config -------------------------------------------------
RUN_NAME       = "maize_expert_v1"
DATA_ROOT      = utils.data_dir()
ARTIFACT_DIR   = utils.artifact_dir()
BACKBONE       = "mobilenetv3"      # "mobilenetv3" | "efficientnetb0" | "resnet50"
BATCH_SIZE     = 32
HEAD_EPOCHS    = 12
FINETUNE_EPOCHS = 8
LEARNING_RATE  = 1e-3
SEED           = 42

utils.seed_everything(SEED)
print("run      :", RUN_NAME)
print("data root:", DATA_ROOT)
print("artifacts:", ARTIFACT_DIR)

In [ ]:
train_dir = utils.find_split_dir(DATA_ROOT, "train")
class_names = utils.list_class_dirs(train_dir)
num_classes = len(class_names)

train_ds, valid_ds = utils.make_train_val_datasets(
    DATA_ROOT, class_names, batch_size=BATCH_SIZE, seed=SEED
)

print(f"{num_classes} classes: {class_names}")
for images, labels in train_ds.take(1):
    print("one batch:", images.shape, labels.shape)

In [ ]:
import tensorflow as tf

# Only backbones that do their own scaling on raw 0-255 pixels are listed here.
# ResNet50/VGG expect preprocessed input and would silently mis-train unless you
# add a matching preprocessing layer in build_model below.
backbones = {
    "mobilenetv3": tf.keras.applications.MobileNetV3Small,
    "efficientnetb0": tf.keras.applications.EfficientNetB0,
}
build_backbone = backbones[BACKBONE]
have_gpu = bool(tf.config.list_physical_devices("GPU"))
print("GPU available:", have_gpu)


def build_model() -> tuple[tf.keras.Model, tf.keras.Model]:
    """Returns (full model, backbone). Backbone expects raw 0-255 pixels."""
    backbone = build_backbone(
        weights="imagenet",
        include_top=False,
        input_shape=(*utils.IMG_SIZE, 3),
    )
    backbone.trainable = False

    inputs = tf.keras.Input(shape=(*utils.IMG_SIZE, 3), name="image")
    x = utils.augmentation_layer()(inputs)
    x = backbone(x, training=False)          # backbone does its own rescaling
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="disease")(x)

    return tf.keras.Model(inputs, outputs, name=RUN_NAME), backbone


model, backbone = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        ARTIFACT_DIR / f"{RUN_NAME}_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=4, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1
    ),
]

history_head = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=HEAD_EPOCHS,
    callbacks=callbacks,
)
model.save(ARTIFACT_DIR / f"{RUN_NAME}_head.keras")

In [ ]:
# Optional fine-tune: unfreeze the last block of the backbone at a low LR.
FINETUNE = True
if FINETUNE:
    backbone.trainable = True
    for layer in backbone.layers[:-20]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(LEARNING_RATE / 50),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    history_ft = model.fit(
        train_ds,
        validation_data=valid_ds,
        epochs=FINETUNE_EPOCHS,
        callbacks=callbacks,
    )
    model.save(ARTIFACT_DIR / f"{RUN_NAME}.keras")
else:
    model.save(ARTIFACT_DIR / f"{RUN_NAME}.keras")

In [ ]:
utils.plot_training_history(history_head, title="Head training")

print("Validation metrics after training:")
for name, value in model.evaluate(valid_ds, verbose=0, return_dict=True).items():
    print(f"   {name:<12} {value:.4f}")

print()
print(utils.classification_report_arrays(model, valid_ds, class_names))

In [ ]:
# Ship the class order with the weights — this is what prevents
# silent label mismatches in the FastAPI service.
names_path = utils.save_class_names(
    ARTIFACT_DIR / f"{RUN_NAME}.class_names.json", class_names
)
print("saved weights :", ARTIFACT_DIR / f"{RUN_NAME}.keras")
print("saved classes :", names_path)

sample_path, _ = utils.list_images(utils.find_split_dir(DATA_ROOT, "valid"))[0]
label, confidence = utils.predict_with_serving_preprocess(model, sample_path, class_names)
print(f"serving-path check on {sample_path.name}: {label} ({confidence:.3f})")

### Next

Open `03_export_and_validate.ipynb` to convert the saved `.keras` file to
TFLite/ONNX, benchmark it, and validate that the exported graph returns the
same predictions as the Keras model.